# 72 - SNP directos MLP_A (Nested CV 5x3) - 7 traits

## Objetivo
Evaluar MLP tabular (solo `mlp_a`) con **Nested CV 5x3** sobre SNPs directos del `.raw`, para los 7 traits del proyecto.

Notas clave:
- Arquitectura fija (no se tunean dropouts ni L2 en esta versiÃ³n).
- Se **tunea solo `learning_rate` y `batch_size`** con inner CV.
- Resultados guardados en `results/...`.


In [1]:
# 1) LibrerÃ­as y parÃ¡metros globales
from pathlib import Path
import re
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import pearsonr

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
except Exception as exc:
    raise ImportError(
        "TensorFlow/Keras no estÃ¡ disponible en este entorno. Instala tensorflow para ejecutar este notebook."
    ) from exc

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# =========================
# Rutas relativas
# =========================
RAW_REL = Path("merged_all_mind0.20_g0.10_mac1_nomaf_pruned_raw.raw")
PHENO_REL = Path("pheno_data/BLUEs_across_env.tsv")
META_REL = Path("pheno_data/giae121_supplemental_file.xlsx")
META_HEADER_ROW = 2

OUTPUT_REL = Path("results/nn_snp_nested5x3")
CHECKPOINT_REL = OUTPUT_REL / "checkpoints"

# =========================
# Traits
# =========================
TRAITS = [
    "blumeria_graminis",
    "heading_date",
    "lodging",
    "plant_height",
    "puccinia_hordei",
    "ramularia_collo_cygni",
    "rhynchosporium",
]

# =========================
# Nested CV
# =========================
OUTER_SPLITS = 5
INNER_SPLITS = 3
MAX_EPOCHS = 300
INNER_VAL_SIZE = 0.20  # para entrenamiento final en outer

# =========================
# Config grid (solo lr y batch)
# =========================
# PequeÃ±o grid por trait con base en los pilotos:
# - lodging: lr=1e-3, batch=16 (mejor en 70)
# - ramularia: lr=1e-3, batch=32 (mejor en 71)
# - otros: grid pequeÃ±o por estabilidad
CONFIG_GRID_BY_TRAIT = {
    "lodging": [
        {"learning_rate": 1e-3, "batch_size": 16},
        {"learning_rate": 1e-3, "batch_size": 32},
    ],
    "ramularia_collo_cygni": [
        {"learning_rate": 1e-3, "batch_size": 32},
        {"learning_rate": 1e-3, "batch_size": 16},
    ],
}

DEFAULT_GRID = [
    {"learning_rate": 1e-3, "batch_size": 32},
    {"learning_rate": 5e-4, "batch_size": 32},
]

EARLY_STOP_PATIENCE = 30
REDUCE_LR_PATIENCE = 10
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-6

VERBOSE_FIT = 0

RAW_ID_COLS = ["FID", "IID", "PAT", "MAT", "SEX", "PHENOTYPE"]

print("TensorFlow:", tf.__version__)


TensorFlow: 2.18.0


In [2]:
# 2) ResoluciÃ³n de rutas y salida (dentro de nested5x3_unificados)

def resolve_base_dir():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "nested5x3_unificados",
        cwd.parent,
        cwd.parent / "nested5x3_unificados",
    ]
    for base in candidates:
        if (base / RAW_REL).exists() and (base / PHENO_REL).exists() and (base / META_REL).exists():
            return base
    raise FileNotFoundError(
        "No se pudieron resolver las rutas de RAW/PHENO/META. Ejecuta desde la raÃ­z o desde nested5x3_unificados."
    )

BASE_DIR = resolve_base_dir()
RAW_PATH = BASE_DIR / RAW_REL
PHENO_PATH = BASE_DIR / PHENO_REL
META_PATH = BASE_DIR / META_REL

OUTPUT_DIR = BASE_DIR / OUTPUT_REL
CHECKPOINT_DIR = BASE_DIR / CHECKPOINT_REL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


BASE_DIR: C:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados
OUTPUT_DIR: C:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados\results\nn_snp_nested5x3


In [3]:
# 3) Carga .raw y SNPs
if not RAW_PATH.exists():
    raise FileNotFoundError(f"No se encuentra el .raw: {RAW_PATH}")

raw = pd.read_csv(RAW_PATH, sep=r"\s+")

if "IID" not in raw.columns:
    raise KeyError("La columna IID no existe en el .raw")

ids = raw["IID"].astype(str).str.strip().rename("IID")
if ids.duplicated().any():
    raise ValueError("Hay IID duplicados en el .raw. Resolver antes de modelar.")

snp_cols = [c for c in raw.columns if c not in RAW_ID_COLS]
if not snp_cols:
    raise ValueError("No se detectaron columnas SNP en el .raw")

X_snp = raw[snp_cols].apply(pd.to_numeric, errors="coerce")

print(f"SNP matrix (n x p): {X_snp.shape}")
print(f"IID Ãºnicos: {ids.nunique()} / {len(ids)}")


SNP matrix (n x p): (1110, 61139)
IID ?nicos: 1110 / 1110


In [4]:
# 4) Merge genotipo-fenotipo (misma lÃ³gica que 53_snp)
if not PHENO_PATH.exists():
    raise FileNotFoundError(f"No se encuentra fenotipo: {PHENO_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"No se encuentra metadata: {META_PATH}")

na_vals = ["", "NA", "NaN", "na", "N/A", "null", "NULL"]
df_pheno = pd.read_csv(PHENO_PATH, sep="	", na_values=na_vals)
meta = pd.read_excel(META_PATH, header=META_HEADER_ROW)
meta = meta.loc[:, ~meta.columns.astype(str).str.contains(r"^Unnamed", regex=True)].copy()

if "genotypes" not in df_pheno.columns:
    raise KeyError("En fenotipo no existe la columna 'genotypes'")

# Resolver IID en metadata
iid_candidates = ["IID", "BioSamples ID", "BioSample ID", "biosamples id", "sample_id", "Sample_ID"]
rename_map = {}
for c in iid_candidates:
    if c in meta.columns:
        rename_map[c] = "IID"
        break
meta = meta.rename(columns=rename_map)
if "IID" not in meta.columns:
    raise KeyError("No se encontrÃ³ columna IID en metadata")


def norm_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    x = re.sub(r"[^A-Z0-9]+", "", x)
    return x if x else np.nan


meta["IID"] = meta["IID"].astype(str).str.strip()
df_pheno["genotypes"] = df_pheno["genotypes"].astype(str).str.strip()
df_pheno["GENO_NORM"] = df_pheno["genotypes"].apply(norm_name)

candidate_name_cols = ["genotype", "genotype name used for BioSamples", "Genebank name"]
meta_name_cols = [c for c in candidate_name_cols if c in meta.columns]
if not meta_name_cols:
    raise KeyError("No hay columnas Ãºtiles para construir puente genotipo -> IID")

bridge_parts = []
for col in meta_name_cols:
    tmp = meta[["IID", col]].copy().rename(columns={col: "GENO_RAW"})
    bridge_parts.append(tmp)

bridge_df = pd.concat(bridge_parts, ignore_index=True)
bridge_df["GENO_NORM"] = bridge_df["GENO_RAW"].apply(norm_name)
bridge_df = bridge_df.dropna(subset=["IID", "GENO_NORM"]).drop_duplicates(["IID", "GENO_NORM"])

trait_cols_present = [t for t in TRAITS if t in df_pheno.columns]
missing_traits = [t for t in TRAITS if t not in df_pheno.columns]
if missing_traits:
    raise ValueError(f"Faltan traits en fenotipo: {missing_traits}")

for t in trait_cols_present:
    df_pheno[t] = pd.to_numeric(df_pheno[t], errors="coerce")

pheno_unique = (
    df_pheno.dropna(subset=["GENO_NORM"])
    .groupby("GENO_NORM", as_index=False)[trait_cols_present]
    .mean()
)

iid_pheno = bridge_df[["IID", "GENO_NORM"]].merge(pheno_unique, on="GENO_NORM", how="left")
iid_pheno = iid_pheno.drop_duplicates(subset=["IID"])
iid_pheno = iid_pheno[["IID"] + trait_cols_present].copy()

snp_df = pd.concat([ids, X_snp], axis=1)
model_df = snp_df.merge(iid_pheno, on="IID", how="left")

feature_cols = snp_cols.copy()
traits_disponibles = [t for t in TRAITS if t in model_df.columns]
print("Traits disponibles:", traits_disponibles)
print("model_df shape:", model_df.shape)


Traits disponibles: ['blumeria_graminis', 'heading_date', 'lodging', 'plant_height', 'puccinia_hordei', 'ramularia_collo_cygni', 'rhynchosporium']
model_df shape: (1110, 61147)


In [5]:
# 5) Funciones auxiliares

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson_r = np.nan
    else:
        pearson_r = float(pearsonr(y_true, y_pred)[0])

    return {
        "pearson_r": pearson_r,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
    }


def build_mlp_a(input_dim, learning_rate):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.35),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.20),
        layers.Dense(1, activation="linear"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"],
    )
    return model


def preprocess_fold_data(X_train_raw, X_val_raw, X_test_raw):
    imputer = SimpleImputer(strategy="mean")
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(X_train_raw)
    X_val_imp = imputer.transform(X_val_raw)
    X_test_imp = imputer.transform(X_test_raw)

    X_train = scaler.fit_transform(X_train_imp).astype(np.float32)
    X_val = scaler.transform(X_val_imp).astype(np.float32)
    X_test = scaler.transform(X_test_imp).astype(np.float32)

    return X_train, X_val, X_test


def fit_model(X_train, y_train, X_val, y_val, lr, batch_size, checkpoint_path=None):
    tf.keras.backend.clear_session()
    model = build_mlp_a(input_dim=X_train.shape[1], learning_rate=lr)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE,
            min_lr=MIN_LR,
        ),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=VERBOSE_FIT,
    )

    history_dict = {k: [float(vv) for vv in vals] for k, vals in history.history.items()}
    best_epoch = int(np.argmin(history_dict["val_loss"]) + 1)
    best_val_loss = float(np.min(history_dict["val_loss"]))

    # Usamos los pesos restaurados por EarlyStopping (restore_best_weights=True)
    return model, history_dict, best_epoch, best_val_loss


In [6]:
# 6) Nested CV 5x3
start_time = datetime.now()
print("Inicio entrenamiento:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

fold_records = []
inner_summary_records = []

best_model_by_trait = {}

for trait in traits_disponibles:
    y_series = pd.to_numeric(model_df[trait], errors="coerce")
    mask = y_series.notna()

    X_trait = model_df.loc[mask, feature_cols].to_numpy(dtype=np.float32, copy=True)
    y_trait = y_series.loc[mask].to_numpy(dtype=np.float32, copy=True)
    iid_trait = model_df.loc[mask, "IID"].astype(str).to_numpy()

    outer_cv = KFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=SEED)
    grid = CONFIG_GRID_BY_TRAIT.get(trait, DEFAULT_GRID)

    print("-" * 80)
    print(f"Trait: {trait} | n={len(y_trait)} | grid={len(grid)}")

    for fold, (outer_tr_idx, outer_te_idx) in enumerate(outer_cv.split(X_trait), start=1):
        X_outer_tr = X_trait[outer_tr_idx]
        y_outer_tr = y_trait[outer_tr_idx]
        X_outer_te = X_trait[outer_te_idx]
        y_outer_te = y_trait[outer_te_idx]
        iid_outer_te = iid_trait[outer_te_idx]

        # Inner CV para selecciÃ³n de hiperparÃ¡metros
        inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + fold)

        cfg_scores = []
        for cfg in grid:
            lr = cfg["learning_rate"]
            bs = cfg["batch_size"]
            inner_metrics = []

            for inner_tr_idx, inner_va_idx in inner_cv.split(X_outer_tr):
                X_in_tr = X_outer_tr[inner_tr_idx]
                y_in_tr = y_outer_tr[inner_tr_idx]
                X_in_va = X_outer_tr[inner_va_idx]
                y_in_va = y_outer_tr[inner_va_idx]

                X_tr, X_va, _ = preprocess_fold_data(X_in_tr, X_in_va, X_in_va)

                model, _, _, _ = fit_model(X_tr, y_in_tr, X_va, y_in_va, lr, bs)
                y_pred_va = model.predict(X_va, batch_size=bs, verbose=0).ravel()
                m = regression_metrics(y_in_va, y_pred_va)
                inner_metrics.append(m)

            # agregaciÃ³n inner
            pearson_mean = np.nanmean([m["pearson_r"] for m in inner_metrics])
            r2_mean = np.mean([m["r2"] for m in inner_metrics])
            rmse_mean = np.mean([m["rmse"] for m in inner_metrics])
            mae_mean = np.mean([m["mae"] for m in inner_metrics])

            cfg_scores.append({
                "trait": trait,
                "outer_fold": fold,
                "learning_rate": lr,
                "batch_size": bs,
                "pearson_r_mean": pearson_mean,
                "r2_mean": r2_mean,
                "rmse_mean": rmse_mean,
                "mae_mean": mae_mean,
            })

        cfg_scores_df = pd.DataFrame(cfg_scores)
        cfg_scores_df = cfg_scores_df.sort_values(
            by=["pearson_r_mean", "r2_mean", "rmse_mean"],
            ascending=[False, False, True]
        )
        best_cfg = cfg_scores_df.iloc[0]

        inner_summary_records.append(best_cfg.to_dict())

        # Entrenamiento final en outer train con split interno
        tr_idx, va_idx = train_test_split(
            np.arange(len(X_outer_tr)),
            test_size=INNER_VAL_SIZE,
            random_state=SEED + fold,
            shuffle=True,
        )

        X_in_tr = X_outer_tr[tr_idx]
        y_in_tr = y_outer_tr[tr_idx]
        X_in_va = X_outer_tr[va_idx]
        y_in_va = y_outer_tr[va_idx]

        X_tr, X_va, X_te = preprocess_fold_data(X_in_tr, X_in_va, X_outer_te)

        model, history_dict, best_epoch, best_val_loss = fit_model(
            X_tr, y_in_tr, X_va, y_in_va,
            best_cfg["learning_rate"],
            int(best_cfg["batch_size"]),
        )

        y_pred_te = model.predict(X_te, batch_size=int(best_cfg["batch_size"]), verbose=0).ravel()
        metrics = regression_metrics(y_outer_te, y_pred_te)

        # Guardar SOLO el mejor checkpoint por trait
        curr = {"pearson_r": metrics["pearson_r"], "r2": metrics["r2"]}
        prev = best_model_by_trait.get(trait)
        is_better = (prev is None) or (
            (curr["pearson_r"] > prev["pearson_r"]) or
            (np.isclose(curr["pearson_r"], prev["pearson_r"]) and curr["r2"] > prev["r2"])
        )
        if is_better:
            best_path = CHECKPOINT_DIR / trait / "best_model.keras"
            best_path.parent.mkdir(parents=True, exist_ok=True)
            model.save(best_path, overwrite=True)
            best_model_by_trait[trait] = {"pearson_r": curr["pearson_r"], "r2": curr["r2"], "path": str(best_path)}

        fold_records.append({
            "trait": trait,
            "fold": fold,
            "learning_rate": float(best_cfg["learning_rate"]),
            "batch_size": int(best_cfg["batch_size"]),
            "pearson_r": float(metrics["pearson_r"]) if pd.notna(metrics["pearson_r"]) else np.nan,
            "r2": float(metrics["r2"]),
            "rmse": float(metrics["rmse"]),
            "mae": float(metrics["mae"]),
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val_loss),
        })

        print(
            f"Trait={trait} | Fold={fold} | lr={best_cfg['learning_rate']} | bs={int(best_cfg['batch_size'])} | "
            f"r={metrics['pearson_r']:.4f} | R2={metrics['r2']:.4f}"
        )

end_time = datetime.now()
print("Fin entrenamiento:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("DuraciÃ³n total:", end_time - start_time)

fold_results_df = pd.DataFrame(fold_records)
inner_best_df = pd.DataFrame(inner_summary_records)


Inicio entrenamiento: 2026-03-29 23:17:38
--------------------------------------------------------------------------------
Trait: blumeria_graminis | n=1110 | grid=2

Trait=blumeria_graminis | Fold=1 | lr=0.001 | bs=32 | r=0.7485 | R2=0.5435
Trait=blumeria_graminis | Fold=2 | lr=0.001 | bs=32 | r=0.7285 | R2=0.5285
Trait=blumeria_graminis | Fold=3 | lr=0.001 | bs=32 | r=0.7578 | R2=0.5653
Trait=blumeria_graminis | Fold=4 | lr=0.001 | bs=32 | r=0.7660 | R2=0.5447
Trait=blumeria_graminis | Fold=5 | lr=0.001 | bs=32 | r=0.7117 | R2=0.4955
--------------------------------------------------------------------------------
Trait: heading_date | n=1110 | grid=2
Trait=heading_date | Fold=1 | lr=0.001 | bs=32 | r=0.8398 | R2=0.7031
Trait=heading_date | Fold=2 | lr=0.001 | bs=32 | r=0.7749 | R2=0.6000
Trait=heading_date | Fold=3 | lr=0.001 | bs=32 | r=0.7534 | R2=0.5639
Trait=heading_date | Fold=4 | lr=0.001 | bs=32 | r=0.7943 | R2=0.5883
Trait=heading_date | Fold=5 | lr=0.0005 | bs=32 | r=0.8278 

In [7]:
# 7) ExportaciÃ³n de resultados
RESULTS_CSV = OUTPUT_DIR / "nn_snp_nested5x3_fold_results.csv"
SUMMARY_CSV = OUTPUT_DIR / "nn_snp_nested5x3_summary.csv"

fold_results_df.to_csv(RESULTS_CSV, index=False)

summary = (
    fold_results_df
    .groupby("trait", as_index=False)
    .agg(
        pearson_r_mean=("pearson_r", "mean"),
        pearson_r_std=("pearson_r", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        best_epoch_mean=("best_epoch", "mean"),
    )
)
summary.to_csv(SUMMARY_CSV, index=False)

print("Guardado:")
print("-", RESULTS_CSV)
print("-", SUMMARY_CSV)


Guardado:
- C:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados\results\nn_snp_nested5x3\nn_snp_nested5x3_fold_results.csv
- C:\Users\Reyes\OneDrive\PC_REYES\Master_Big_Data_Data_Science\14MBID\2026\Code\nested5x3_unificados\results\nn_snp_nested5x3\nn_snp_nested5x3_summary.csv
